# 02 — Classificação com árvore de decisão de raiz
**Objetivo proposto:** prever dificuldades de pagamento, conforme TARGET no dicionário.

A árvore está em libs/decisiontrees.py. Não usamos classificadores de bibliotecas de ML.
pandas está autorizado provisoriamente pelo grupo. Confirmar com o docente as regras para NumPy/Matplotlib; os algoritmos são implementados de raiz.

**Estado:** implementação inicial, sem resultados reais publicados.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from libs.data import FEATURES, load_applications

SEED = 42
# Mantém downloads explícitos: a autenticação e as regras são geridas no Kaggle.
DOWNLOAD = False
applications, dictionary = load_applications(download=DOWNLOAD)
print(f"Pedidos: {len(applications):,}; colunas: {len(applications.columns)}")

# Guardar estas informações com os resultados permite repetir a experiência.
display(pd.DataFrame(applications.attrs["source_files"]))
print({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__})

## Amostra e separação
O limite inicial de 20 000 linhas é um orçamento de desenvolvimento, não uma conclusão
sobre o tamanho ideal. A amostra aleatória não equilibra artificialmente as classes.
Registar a dimensão efetiva e rever o limite após medir o tempo.

Dentro de cada classe, separamos aproximadamente 60% treino, 20% validação e 20% teste.
O teste fica reservado para a avaliação final.

In [ ]:
from libs.data import NumericPreparation, stratified_split
from libs.decisiontrees import DecisionTree
from libs.metrics import classification_metrics
from itertools import product
from time import perf_counter

MAX_ROWS = 20_000
sample = applications.sample(n=min(MAX_ROWS, len(applications)), random_state=SEED)
train, validation, test = stratified_split(sample, seed=SEED)
display(pd.DataFrame({
    name: frame["TARGET"].value_counts().sort_index()
    for name, frame in [("treino", train), ("validação", validation), ("teste", test)]
}).fillna(0).astype(int))
preparation = NumericPreparation().fit(train)
X_train = preparation.transform(train)
X_validation = preparation.transform(validation)
y_train = train["TARGET"].to_numpy()
y_validation = validation["TARGET"].to_numpy()

## Funcionamento
Em cada nó, ordenamos cada variável, avaliamos as divisões entre valores distintos e
escolhemos a menor impureza de Gini ponderada. Uma folha guarda a proporção de TARGET=1.

Hiperparâmetros: max_depth, min_samples_split e min_samples_leaf.
A validação escolhe também um limiar de decisão entre 0.1, 0.2 e 0.5.
Um limiar menor pode melhorar recall à custa de mais falsos positivos.
A grelha abaixo é inicial; não representa valores já demonstrados como ótimos.

In [ ]:
results = []
best_model = None
best_threshold = None
best_f1 = -1.0
started = perf_counter()
for depth, split, leaf in product([3, 5, 7], [20, 100], [10, 50]):
    model = DecisionTree(depth, split, leaf).fit(X_train, y_train)
    probabilities = model.predict_proba(X_validation)
    for threshold in [0.1, 0.2, 0.5]:
        metrics = classification_metrics(y_validation, (probabilities > threshold).astype(int))
        results.append({
            "max_depth": depth, "min_samples_split": split,
            "min_samples_leaf": leaf, "threshold": threshold, **metrics,
        })
        # Em empate, mantém a primeira configuração: profundidade menor vem primeiro.
        if metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            best_model, best_threshold = model, threshold
validation_results = pd.DataFrame(results).sort_values("f1", ascending=False, kind="stable")
print(f"Tempo da pesquisa: {perf_counter() - started:.1f} s")
display(validation_results)

## Avaliação final
Mantemos o modelo treinado no treino e escolhido na validação.
O limiar não é reajustado pelo teste. A baseline prevê a classe maioritária do treino.
Precisão e F1 valem zero quando não existem previsões positivas.

In [ ]:
X_test = preparation.transform(test)
y_test = test["TARGET"].to_numpy()
majority = int(y_train.mean() > 0.5)
baseline = np.full(len(test), majority)
predictions = (best_model.predict_proba(X_test) > best_threshold).astype(int)
final_metrics = classification_metrics(y_test, predictions)
display(pd.DataFrame({
    "baseline": classification_metrics(y_test, baseline),
    "árvore": final_metrics,
}).T)
display(pd.DataFrame(
    [[final_metrics["tn"], final_metrics["fp"]], [final_metrics["fn"], final_metrics["tp"]]],
    index=["Real 0", "Real 1"], columns=["Previsto 0", "Previsto 1"],
))
print("Limiar escolhido na validação:", best_threshold)

## Interpretação — preencher após executar
- A árvore supera a baseline? Em que métricas?
- Quantos pedidos com dificuldades ficaram por identificar?
- Qual o compromisso entre falsos positivos e falsos negativos?
- Que configurações revelam sobreajuste ou pouco ganho?
- Que limitações vêm da amostra, das sete variáveis e da imputação?

Os resultados descrevem esta experiência académica e não validam uma política de concessão de crédito.